# HelpMateAI_News Analysis

HelpMateAI_News Analysis is a Retrieval-Augmented Generation (RAG) system designed to assist news professionals in quickly retrieving and summarizing key information from  documents. The system allows users to ask questions and receive accurate, context-aware responses, significantly reducing research time and improving efficiency.
This system leverages AI to assist users in processing complex documents, ensuring that critical information is accessible within seconds.


Problem Statement is in document file

In [ ]:
# Installing the requirement

In [2]:
!pip install langchain groq pypdf chromadb transformers sentence-transformers

In [3]:
!pip install -U langchain-community

In [4]:
pip install -U langchain-groq

In [5]:
pip install -qU "langchain[groq]"

In [6]:
pip install pymupdf

In [ ]:
Importing necessary Libraries

In [2]:
from google.colab import drive
import os
import json

In [3]:
import os
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.vectorstores import Chroma
from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_groq import ChatGroq
from langchain.chains import RetrievalQA
from langchain.retrievers import MultiQueryRetriever
from langchain.schema import SystemMessage, HumanMessage, AIMessage

In [ ]:
# Mounting Google drive

In [ ]:
!rm -rf /root/.config/Google/DriveFS


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount = True)

Mounted at /content/drive


In [ ]:
# Creating JSON file and  storing api keys

In [28]:
'''
# Define project folder
project_folder = "/content/drive/MyDrive/myproject"
os.makedirs(project_folder, exist_ok=True)

# Define  API keys (Replace with actual keys)
api_keys = {
    "GROQ_API_KEY": "your_actual_groq_api_key_here",
    "HUGGINGFACE_API_KEY": "your_actual_huggingface_api_key_here",

}

# Save API keys to a JSON file
with open(f"{project_folder}/api_keys.json", "w") as f:
    json.dump(api_keys, f)

print( "API keys saved securely! You can remove api and comment out or delete this cell now.")

API keys saved securely! You can remove api and comment out or delete this cell now.


In [ ]:
# Passing Api keys

In [30]:

# Load API keys from JSON file
with open("/content/drive/MyDrive/myproject/api_keys.json", "r") as f:
    api_keys = json.load(f)

'''# Store API keys as environment variables
for key, value in api_keys.items():
    os.environ[key] = value

print(" API keys loaded securely!")'''

# Access individual keys
groq_api_key = api_keys["GROQ_API_KEY"]
huggingface_api_key = api_keys["HUGGINGFACE_API_KEY"]


In [ ]:
# Creating data folder

In [5]:
import os

pdf_folder = "/content/drive/MyDrive/myproject/data"

# Create the folder if it doesn't exist
if not os.path.exists(pdf_folder):
    os.makedirs(pdf_folder)
    print(f"Created folder: {pdf_folder}")
else:
    print(f" Folder exists: {pdf_folder}")


 Folder exists: /content/drive/MyDrive/myproject/data


In [ ]:
# Loding pdf file

In [6]:
from langchain.document_loaders import PyMuPDFLoader
import os

pdf_folder = "/content/drive/MyDrive/myproject/data"
documents = [doc for pdf in os.listdir(pdf_folder) if pdf.endswith(".pdf")
             for doc in PyMuPDFLoader(os.path.join(pdf_folder, pdf)).load()]
# Add metadata (e.g., source)
for doc in documents:
    doc.metadata['source'] = pdf_folder  # Additional metadata can be added here
pdf_files = [f for f in os.listdir(pdf_folder) if f.endswith(".pdf")]

print(f"Found {len(pdf_files)} PDFs:", pdf_files)
print(f"Loaded {len(documents)} pages with metadata.")
print("Metadata Example:", documents[0].metadata)

Found 1 PDFs: ['4_Executive-Summary-067bd6a9cef9eb4.20015552.pdf']
Loaded 8 pages with metadata.
Metadata Example: {'producer': 'Adobe Acrobat Pro DC (32-bit) 21.5.20060', 'creator': 'Adobe Acrobat Pro DC (32-bit) 21.5.20060', 'creationdate': '2024-03-06T10:24:19+05:30', 'source': '/content/drive/MyDrive/myproject/data', 'file_path': '/content/drive/MyDrive/myproject/data/4_Executive-Summary-067bd6a9cef9eb4.20015552.pdf', 'total_pages': 8, 'format': 'PDF 1.6', 'title': '', 'author': 'Deepak G', 'subject': '', 'keywords': '', 'moddate': '2024-03-06T10:24:23+05:30', 'trapped': '', 'modDate': "D:20240306102423+05'30'", 'creationDate': "D:20240306102419+05'30'", 'page': 0}


In [7]:
# reading the data
documents[3]

Document(metadata={'producer': 'Adobe Acrobat Pro DC (32-bit) 21.5.20060', 'creator': 'Adobe Acrobat Pro DC (32-bit) 21.5.20060', 'creationdate': '2024-03-06T10:24:19+05:30', 'source': '/content/drive/MyDrive/myproject/data', 'file_path': '/content/drive/MyDrive/myproject/data/4_Executive-Summary-067bd6a9cef9eb4.20015552.pdf', 'total_pages': 8, 'format': 'PDF 1.6', 'title': '', 'author': 'Deepak G', 'subject': '', 'keywords': '', 'moddate': '2024-03-06T10:24:23+05:30', 'trapped': '', 'modDate': "D:20240306102423+05'30'", 'creationDate': "D:20240306102419+05'30'", 'page': 3}, page_content='Performance Audit on Regulation and Supply of Liquor in Delhi \nx \ndifferent category (Wholesaler, Retailer, HCR etc.) to related parties, leading to the \nexistence of common directorship among entities holding various License Types.  \nFurther the Department was issuing licenses without checking various requirements \nrelating to Excise Rules and Terms and Conditions for the issue of different type

In [8]:
# Splitting the data in to chunk
splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=20)
chunks = splitter.split_documents(documents)

print(f"Created {len(chunks)} text chunks.")

Created 99 text chunks.


In [ ]:
# Clearing the database
import shutil
del db
del retriever

In [10]:
# Loading embedding model
embedding_model = HuggingFaceEmbeddings(model_name="BAAI/bge-small-en")
#embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# Storing in ChromaDB
db = Chroma.from_documents(chunks, embedding_model)
retriever = db.as_retriever()

print(" PDFs stored in ChromaDB!")

<ipython-input-10-f1c7f0b68f7a>:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(model_name="BAAI/bge-small-en")
/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warni

 PDFs stored in ChromaDB!


In [11]:
from langchain.chat_models import init_chat_model

llm_groq = init_chat_model("llama3-8b-8192", model_provider="groq", temperature=0.8, max_tokens=256,)
# Define system instruction with hallucination prevention
system_message = SystemMessage(content=(
    "You are an AI assistant providing concise answers strictly based on the given documents. "
    "If the answer is not found in the documents, do not make up information. "
    "Instead, politely respond: 'I have no answer according to the documents.'"
))
# Define few-shot examples
few_shot_examples = [
    HumanMessage(content="What is the capital of France?"),
    AIMessage(content="The capital of France is Paris."),

    HumanMessage(content="Who wrote 'Hamlet'?"),
    AIMessage(content="William Shakespeare wrote 'Hamlet'."),

    HumanMessage(content="What is the population of Atlantis?"),
    AIMessage(content="I have no answer according to the documents.")
]

In [12]:
qa_chain = RetrievalQA.from_chain_type(llm=llm_groq, retriever=retriever,
    return_source_documents=True)

In [13]:
# Testing the model
query = "what the document representing"
result = qa_chain.invoke(query)

print(" Answer:", result["result"])

if "source_documents" in result:
    for i, doc in enumerate(result["source_documents"]):
        print(f" Source {i+1}: {doc.metadata}")
else:
    print(" No source documents returned.")


 Answer: The document appears to be an Executive Summary, likely representing a tender document or a proposal for a project related to the liquor industry, specifically regarding the licenses and regulations of wholesale and zonal licensees.
 Source 1: {'author': 'Deepak G', 'creationDate': "D:20240306102419+05'30'", 'creationdate': '2024-03-06T10:24:19+05:30', 'creator': 'Adobe Acrobat Pro DC (32-bit) 21.5.20060', 'file_path': '/content/drive/MyDrive/myproject/data/4_Executive-Summary-067bd6a9cef9eb4.20015552.pdf', 'format': 'PDF 1.6', 'keywords': '', 'modDate': "D:20240306102423+05'30'", 'moddate': '2024-03-06T10:24:23+05:30', 'page': 0, 'producer': 'Adobe Acrobat Pro DC (32-bit) 21.5.20060', 'source': '/content/drive/MyDrive/myproject/data', 'subject': '', 'title': '', 'total_pages': 8, 'trapped': ''}
 Source 2: {'author': 'Deepak G', 'creationDate': "D:20240306102419+05'30'", 'creationdate': '2024-03-06T10:24:19+05:30', 'creator': 'Adobe Acrobat Pro DC (32-bit) 21.5.20060', 'file_p

In [14]:
# Refining the model
retriever = db.as_retriever(
    search_type="mmr",  # Maximal Marginal Relevance (Hybrid Search)
    search_kwargs={"k": 3}  # Retrieve top 3 most relevant chunks
)


In [15]:
reranker = MultiQueryRetriever.from_llm(
    retriever=retriever,
    llm=init_chat_model("llama-guard-3-8b", model_provider="groq", temperature=0.8, max_tokens=500),

)


In [16]:
qa_chain = RetrievalQA.from_chain_type(
    llm=init_chat_model("llama3-8b-8192", model_provider="groq", temperature=0.8, max_tokens=256),
    retriever=reranker,
    return_source_documents=True
)


In [32]:
# Testing the model
query = "what is  the Issue in Design and Award of Licenses  ?"
result = qa_chain.invoke(query, return_source_documents=True)

# Extract Answer
answer = result["result"]
source_documents = result["source_documents"]
#print("Answer:", answer)

# Print Metadata from Retrieved Documents
for i, doc in enumerate(source_documents):
    print(f" Source {i+1}: {doc.metadata}")
answer

 Source 1: {'author': 'Deepak G', 'creationDate': "D:20240306102419+05'30'", 'creationdate': '2024-03-06T10:24:19+05:30', 'creator': 'Adobe Acrobat Pro DC (32-bit) 21.5.20060', 'file_path': '/content/drive/MyDrive/myproject/data/4_Executive-Summary-067bd6a9cef9eb4.20015552.pdf', 'format': 'PDF 1.6', 'keywords': '', 'modDate': "D:20240306102423+05'30'", 'moddate': '2024-03-06T10:24:23+05:30', 'page': 0, 'producer': 'Adobe Acrobat Pro DC (32-bit) 21.5.20060', 'source': '/content/drive/MyDrive/myproject/data', 'subject': '', 'title': '', 'total_pages': 8, 'trapped': ''}
 Source 2: {'author': 'Deepak G', 'creationDate': "D:20240306102419+05'30'", 'creationdate': '2024-03-06T10:24:19+05:30', 'creator': 'Adobe Acrobat Pro DC (32-bit) 21.5.20060', 'file_path': '/content/drive/MyDrive/myproject/data/4_Executive-Summary-067bd6a9cef9eb4.20015552.pdf', 'format': 'PDF 1.6', 'keywords': '', 'modDate': "D:20240306102423+05'30'", 'moddate': '2024-03-06T10:24:23+05:30', 'page': 5, 'producer': 'Adobe A

'Based on the provided context, the issue appears to be that Department issued licenses despite major shortcomings, including incomplete or inaccurate test reports on water quality, harmful ingredients, heavy metals, methyl alcohol, and microbiological contaminants.'

In [ ]:
# Creating interactive chat

In [22]:
## Query response function
def query_response(user_input):
    # Query using LangChain's RetrievalQA
    response = qa_chain.invoke(query, return_source_documents=True)

    # Check if source documents are available
    if "source_documents" in response and response["source_documents"]:
        sources = response["source_documents"]

        # Extract file name and pages
        file_name = pdf_files
        page_numbers = ", ".join([doc.metadata.get("source", "Unknown") for doc in sources[:2]])

        # Format final response
        final_response = response["result"] + f"\n Check further at {file_name}, pages {page_numbers}"
    else:
        final_response = response["result"] + "\n No source documents available."

    return final_response


In [23]:
from IPython.core.display import display, HTML  #  Import HTML & display

def initialize_conv():
    print('Feel free to ask questions regarding current news. Type "exit" to quit.')

    while True:
        user_input = input()
        if user_input.lower() == 'exit':
            print('Exiting the program... Bye!')
            break
        else:
            response = query_response(user_input)  # Call the query function
            display(HTML(f'<p style="font-size:16px">{response}</p>'))  #  Display output correctly

# Run the conversation function
initialize_conv()


Feel free to ask questions regarding current news. Type "exit" to quit.
what is  the Issue in Design and Award of Licenses


exit
Exiting the program... Bye!


In [24]:
questions = ['what is  the Issue in Design and Award of Licenses','what is the standard of quality','what is the Issues in implementation of Excise Policy  ?']

In [25]:
import pandas as pd  #  Import pandas

def testing_pipeline(questions):
    test_feedback = []

    for i in questions:
        print(i)
        response = query_response(i)  #  Call the function only once
        print(response)

        print('\nPlease provide your feedback on the response provided by the bot (Good/Bad):')
        user_input = input()

        page = response.split()[-1]  # Extract the last word as the page number
        test_feedback.append((i, response, page, user_input))

    #  Create a DataFrame with proper column names
    feedback_df = pd.DataFrame(test_feedback, columns=['Question', 'Response', 'Page', 'Feedback'])

    return feedback_df  #  Return DataFrame


In [26]:
testing_pipeline(questions)

what is  the Issue in Design and Award of Licenses
Based on the provided context, the issue appears to be that the Department issued licenses despite major shortcomings, without clarifying what those shortcomings are. However, it is mentioned that important test reports on water quality, harmful ingredients, heavy metals, methyl alcohol, and microbiological tests were not taken into account. It can be inferred that the licenses were issued without ensuring the quality and safety of the products or services being licensed.
🔹 Check further at ['4_Executive-Summary-067bd6a9cef9eb4.20015552.pdf'], pages /content/drive/MyDrive/myproject/data, /content/drive/MyDrive/myproject/data

Please provide your feedback on the response provided by the bot (Good/Bad):
good
what is the standard of quality
Based on the provided context, the issue appears to be that the Department issued licenses despite major shortcomings, which are not specified. However, it can be inferred that the licenses were issued

,Question,Response,Page,Feedback
0,what is the Issue in Design and Award of Lice...,"Based on the provided context, the issue appea...",/content/drive/MyDrive/myproject/data,good
1,what is the standard of quality,"Based on the provided context, the issue appea...",/content/drive/MyDrive/myproject/data,good
2,what is the Issues in implementation of Excise...,"Based on the given context, the issue seems to...",/content/drive/MyDrive/myproject/data,good
